# Food Image Recognition — Fine-tune nateraw/food (ViT-B) to 110 Classes

**Run order:**
1. Enable **T4 GPU**: Runtime → Change runtime type → T4
2. **Cell 1** — install (Colab restarts after this, that is normal)
3. **Cell 2** — upload `taiwanese_scraped.zip` when prompted
4. **Cell 3** — streams Food-101 one image at a time (low RAM), builds dataset
5. **Cell 4** — fine-tune
6. **Cell 5** — download `food_vit.zip`

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
# Colab will restart the runtime after this — that is expected.
!pip install -q transformers datasets torch torchvision accelerate

In [ ]:
# ── Cell 2: Upload + unzip Taiwanese images ───────────────────────────────────
from google.colab import files
import zipfile, os

print('Select taiwanese_scraped.zip …')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')

scraped_root = 'data/food_dataset/scraped'
tw_classes = sorted(os.listdir(scraped_root))
print(f'Taiwanese classes ({len(tw_classes)}): {tw_classes}')

In [ ]:
# ── Cell 3: Build dataset (streaming — uses ~200 MB RAM, not 12 GB) ───────────
# Food-101 images are written to disk one at a time via streaming.
# Uses the official Food-101 train/val split (75k train, 25k val).
import shutil, os
from pathlib import Path
from datasets import load_dataset

OUT_DIR      = Path('data/food_dataset')
SCRAPED_ROOT = Path('data/food_dataset/scraped')   # redefined here — no dependency on Cell 2 vars

# ── Step A: get label names (loads only 10 images, fast) ──────────────────────
sample       = load_dataset('food101', split='train[:10]', trust_remote_code=True)
label_names  = sample.features['label'].names
del sample
print(f'Food-101 label names loaded: {len(label_names)} classes')

# ── Step B: stream train split → write one image at a time ────────────────────
for hf_split, our_split in [('train', 'train'), ('validation', 'val')]:
    print(f'Streaming Food-101 {hf_split} → {our_split}/ …')
    ds      = load_dataset('food101', split=hf_split, trust_remote_code=True, streaming=True)
    counts  = {}
    for item in ds:
        cls   = label_names[item['label']]
        d     = OUT_DIR / our_split / cls
        d.mkdir(parents=True, exist_ok=True)
        n     = counts.get(cls, 0)
        item['image'].convert('RGB').save(d / f'{n:04d}.jpg', 'JPEG', quality=85)
        counts[cls] = n + 1
    total = sum(counts.values())
    print(f'  ✓ {total} images written ({len(counts)} classes)')

# ── Step C: copy Taiwanese scraped images (already resized to 256×256) ────────
import random
random.seed(42)

print('Copying Taiwanese classes…')
for cls_dir in sorted(SCRAPED_ROOT.iterdir()):
    if not cls_dir.is_dir():
        continue
    imgs  = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    random.shuffle(imgs)
    split = int(len(imgs) * 0.8)
    for dest, fs in [('train', imgs[:split]), ('val', imgs[split:])]:
        d = OUT_DIR / dest / cls_dir.name
        d.mkdir(parents=True, exist_ok=True)
        for f in fs:
            shutil.copy(f, d / f.name)
    print(f'  {cls_dir.name}: {len(imgs)} images')

train_classes = sorted(p.name for p in (OUT_DIR / 'train').iterdir() if p.is_dir())
print(f'\n✓ Dataset ready: {len(train_classes)} classes')
print(f'  train: {sum(1 for _ in (OUT_DIR/"train").rglob("*.jpg"))} images')
print(f'  val:   {sum(1 for _ in (OUT_DIR/"val").rglob("*.jpg"))} images')

In [ ]:
# ── Cell 4: Fine-tune last layer of nateraw/food (ViT-B/16) ──────────────────
import json, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification
from pathlib import Path
from google.colab import drive

# ── Mount Google Drive ────────────────────────────────────────────────────────
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/food_vit_checkpoint')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

BASE_MODEL  = 'nateraw/food'
DATA_DIR    = Path('data/food_dataset')
SAVE_DIR    = Path('food_vit')          # local copy (also saved to Drive each epoch)
EPOCHS      = 15
BATCH_SIZE  = 512
LR          = 1e-3
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

print('Loading nateraw/food…')
extractor = AutoImageProcessor.from_pretrained(BASE_MODEL)
model     = AutoModelForImageClassification.from_pretrained(BASE_MODEL)

for p in model.parameters():
    p.requires_grad = False

mean, std = extractor.image_mean, extractor.image_std
train_tf  = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

train_ds    = datasets.ImageFolder(DATA_DIR / 'train', transform=train_tf)
val_ds      = datasets.ImageFolder(DATA_DIR / 'val',   transform=val_tf)
num_classes = len(train_ds.classes)
print(f'Classes: {num_classes}  |  train: {len(train_ds)}  |  val: {len(val_ds)}')

# Replace head
num_features      = model.classifier.in_features
model.classifier  = nn.Linear(num_features, num_classes)
model.config.num_labels = num_classes

# ── Resume from Drive checkpoint if one exists ────────────────────────────────
ckpt_file  = DRIVE_DIR / 'trainer_state.json'
start_epoch = 0
best_acc    = 0.0

if ckpt_file.exists():
    state = json.loads(ckpt_file.read_text())
    start_epoch = state['epoch']         # resume from next epoch
    best_acc    = state['best_acc']
    # Load saved head weights
    head_ckpt = DRIVE_DIR / 'head.pt'
    if head_ckpt.exists():
        model.classifier.load_state_dict(torch.load(head_ckpt, map_location='cpu'))
        print(f'Resumed from epoch {start_epoch}  best_acc={best_acc:.3f}')
else:
    print('No checkpoint found — starting fresh')

model.to(DEVICE)

# Save class→idx (always, so inference works even before training finishes)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
class_to_idx_str = json.dumps(train_ds.class_to_idx, ensure_ascii=False, indent=2)
(SAVE_DIR    / 'class_to_idx.json').write_text(class_to_idx_str)
(DRIVE_DIR   / 'class_to_idx.json').write_text(class_to_idx_str)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, last_epoch=start_epoch - 1)
criterion = nn.CrossEntropyLoss()
total_batches = len(train_dl)
print(f'Starting from epoch {start_epoch+1} — {total_batches} batches/epoch, {EPOCHS} epochs total\n')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0.0
    for i, (imgs, labels) in enumerate(train_dl):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs).logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        if (i + 1) % 20 == 0 or (i + 1) == total_batches:
            print(f'  Epoch {epoch+1}/{EPOCHS}  batch {i+1}/{total_batches}  loss={train_loss/(i+1):.4f}')
    scheduler.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    acc = correct / total
    is_best = acc > best_acc
    if is_best:
        best_acc = acc

    print(f'→ Epoch {epoch+1}/{EPOCHS} done  val_acc={acc:.3f}{"  ✓ NEW BEST" if is_best else ""}\n')

    # Save checkpoint to Drive every epoch (safe against disconnection)
    torch.save(model.classifier.state_dict(), DRIVE_DIR / 'head.pt')
    (DRIVE_DIR / 'trainer_state.json').write_text(
        json.dumps({'epoch': epoch + 1, 'best_acc': best_acc})
    )
    if is_best:
        model.save_pretrained(DRIVE_DIR)
        extractor.save_pretrained(DRIVE_DIR)
        model.save_pretrained(SAVE_DIR)
        extractor.save_pretrained(SAVE_DIR)
        print(f'  Best model saved to Drive ({best_acc:.3f})')

print(f'✓ Training done.  Best val_acc: {best_acc:.3f}')
print(f'  Model saved at: {DRIVE_DIR}')

In [ ]:
# ── Cell 5: Download the trained model ───────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive('food_vit', 'zip', '.', 'food_vit')
files.download('food_vit.zip')
print('Downloaded food_vit.zip')
print('→ Extract and place food_vit/ at: backend/cv/models/food_vit/')